In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'kagglehub'], check=True)
print('Done.')

In [ ]:
import os
import json
import math
import random
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

IS_KAGGLE = os.path.exists('/kaggle/working')

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif hasattr(torch, 'mps') and torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print('Device:', DEVICE, '| kaggle:', IS_KAGGLE)

## Данные — Russian Road Signs Segmentation

In [ ]:
import kagglehub

ds_path = kagglehub.dataset_download('viacheslavshalamov/russian-road-signs-segmentation-dataset')
SRC_ROOT = Path(ds_path) / 'sign_dataset'
print('Source root:', SRC_ROOT)

In [ ]:
def sample_ellipse(cx, cy, rx, ry, n_pts=40):
    angles = np.linspace(0, 2 * math.pi, n_pts, endpoint=False)
    xs = cx + rx * np.cos(angles)
    ys = cy + ry * np.sin(angles)
    return list(zip(xs, ys))


def sample_circle(cx, cy, r, n_pts=40):
    return sample_ellipse(cx, cy, r, r, n_pts)


def polygon_to_yolo_seg(class_id, pts, img_w, img_h):
    coords = []
    for px, py in pts:
        px = float(np.clip(px, 0, img_w - 1))
        py = float(np.clip(py, 0, img_h - 1))
        coords.append(f'{px / img_w:.6f}')
        coords.append(f'{py / img_h:.6f}')
    if len(coords) < 6:
        return None
    return str(class_id) + ' ' + ' '.join(coords)

In [ ]:
def build_yolo_split(src_root: Path, split_name: str, out_root: Path, pts_per_curve: int = 40):
    src_split = src_root / split_name
    via_json = json.loads((src_split / 'via_region_data.json').read_text(encoding='utf-8'))

    img_out = out_root / 'images' / split_name
    lbl_out = out_root / 'labels' / split_name
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for entry in via_json.values():
        fname = entry['filename']
        img_src = src_split / fname
        if not img_src.exists():
            continue

        frame = cv2.imread(str(img_src))
        if frame is None:
            continue
        H, W = frame.shape[:2]

        lines = []
        for region in entry.get('regions', {}).values():
            r_attrs = region.get('region_attributes', {})
            if r_attrs.get('name') != 'road sign':
                continue

            shp = region['shape_attributes']
            shape_type = shp.get('name')

            if shape_type == 'polygon':
                pts = list(zip(shp['all_points_x'], shp['all_points_y']))
            elif shape_type == 'ellipse':
                pts = sample_ellipse(shp['cx'], shp['cy'], shp['rx'], shp['ry'], pts_per_curve)
            elif shape_type == 'circle':
                pts = sample_circle(shp['cx'], shp['cy'], shp['r'], pts_per_curve)
            else:
                continue

            line = polygon_to_yolo_seg(0, pts, W, H)
            if line:
                lines.append(line)

        stem = Path(fname).stem
        (lbl_out / f'{stem}.txt').write_text('\n'.join(lines), encoding='utf-8')
        shutil.copy2(img_src, img_out / fname)

    print(f'Built split: {split_name}')

In [ ]:
YOLO_DS = Path('/kaggle/working/road_signs_yolo') if IS_KAGGLE else Path('road_signs_yolo')

if not YOLO_DS.exists():
    for split in ('train', 'val'):
        build_yolo_split(SRC_ROOT, split, YOLO_DS)

    yaml_content = (
        f'path: {YOLO_DS}\n'
        f'train: {YOLO_DS}/images/train\n'
        f'val: {YOLO_DS}/images/val\n'
        'names:\n'
        '  0: road_sign\n'
    )
    (YOLO_DS / 'data.yaml').write_text(yaml_content, encoding='utf-8')
    print('Dataset ready.')
else:
    print('Dataset already prepared.')

### Примеры данных

In [ ]:
import matplotlib.pyplot as plt

val_imgs = list((YOLO_DS / 'images' / 'val').glob('*.jpg'))
sample_imgs = random.sample(val_imgs, min(6, len(val_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flat, sample_imgs):
    img = cv2.imread(str(img_path))
    H, W = img.shape[:2]
    lbl = YOLO_DS / 'labels' / 'val' / f'{img_path.stem}.txt'

    overlay = img.copy()
    if lbl.exists():
        for ln in lbl.read_text(encoding='utf-8').splitlines():
            if not ln.strip():
                continue
            vals = list(map(float, ln.split()[1:]))
            pts = np.array(vals, dtype=np.float32).reshape(-1, 2)
            pts[:, 0] *= W
            pts[:, 1] *= H
            pts = pts.astype(np.int32)
            cv2.fillPoly(overlay, [pts], (50, 200, 50))
            cv2.polylines(img, [pts], True, (0, 50, 220), 2)

    blended = cv2.addWeighted(overlay, 0.3, img, 0.7, 0)
    ax.imshow(cv2.cvtColor(blended, cv2.COLOR_BGR2RGB))
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Обучение модели — Stage 1

In [ ]:
DATA_YAML = YOLO_DS / 'data.yaml'

IMG_SIZE = 512
BATCH = 8 if IS_KAGGLE else 2

S1_PROJECT = '/kaggle/working/seg_runs' if IS_KAGGLE else 'seg_runs'
S1_RUN = 'stage1'
s1_last = Path(S1_PROJECT) / S1_RUN / 'weights' / 'last.pt'
s1_best = Path(S1_PROJECT) / S1_RUN / 'weights' / 'best.pt'

if s1_last.exists():
    seg_model = YOLO(str(s1_last))
    try:
        seg_model.train(resume=True)
    except AssertionError:
        print('Stage 1 already done.')
else:
    seg_model = YOLO('yolo11n-seg.pt')
    seg_model.train(
        data=str(DATA_YAML),
        epochs=50,
        imgsz=IMG_SIZE,
        batch=BATCH,
        device=DEVICE,
        workers=0,
        project=S1_PROJECT,
        name=S1_RUN,
        exist_ok=True,
        cache=False,
        seed=SEED,
        fraction=0.2,
        amp=False,
        mosaic=0.0,
        mixup=0.0,
        copy_paste=0.0,
    )

## Stage 2 — дообучение на полном датасете

In [ ]:
S2_RUN = 'stage2'
s2_last = Path(S1_PROJECT) / S2_RUN / 'weights' / 'last.pt'
s2_best = Path(S1_PROJECT) / S2_RUN / 'weights' / 'best.pt'

if s2_last.exists():
    seg_model2 = YOLO(str(s2_last))
    try:
        seg_model2.train(resume=True)
    except AssertionError:
        print('Stage 2 already done.')
else:
    seg_model2 = YOLO(str(s1_best))
    seg_model2.train(
        data=str(DATA_YAML),
        epochs=50,
        imgsz=IMG_SIZE,
        batch=BATCH,
        device=DEVICE,
        workers=0,
        project=S1_PROJECT,
        name=S2_RUN,
        exist_ok=True,
        cache=False,
        seed=SEED,
        fraction=1.0,
        amp=False,
        mosaic=0.0,
        mixup=0.0,
        copy_paste=0.0,
    )

## Stage 3 — высокое разрешение

In [ ]:
S3_RUN = 'stage3_hires'
s3_last = Path(S1_PROJECT) / S3_RUN / 'weights' / 'last.pt'
s3_best = Path(S1_PROJECT) / S3_RUN / 'weights' / 'best.pt'

if s3_last.exists():
    seg_model3 = YOLO(str(s3_last))
    try:
        seg_model3.train(resume=True)
    except AssertionError:
        print('Stage 3 already done.')
else:
    seg_model3 = YOLO(str(s2_best))
    seg_model3.train(
        data=str(DATA_YAML),
        epochs=20,
        imgsz=1280,
        batch=BATCH,
        device=DEVICE,
        workers=0,
        project=S1_PROJECT,
        name=S3_RUN,
        exist_ok=True,
        cache=False,
        seed=SEED,
        fraction=0.1,
        amp=False,
        mosaic=0.0,
        mixup=0.0,
        copy_paste=0.0,
    )

## Инференс на тестовых изображениях

In [ ]:
def run_inference(weights, run_tag, imgsz=512, conf_thr=0.35, nms_iou=0.3):
    mdl = YOLO(weights)
    mdl.predict(
        source=sorted([str(p) for p in (Path('/kaggle/input/hw3-media') if IS_KAGGLE else Path('test_images')).glob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]),
        device=DEVICE,
        imgsz=imgsz,
        conf=conf_thr,
        iou=nms_iou,
        retina_masks=True,
        save=True,
        project=str(Path('/kaggle/working/test_results') if IS_KAGGLE else Path('test_results')),
        name=run_tag,
        exist_ok=True,
        line_width=2,
    )
    print('Saved to test_results/', run_tag)


run_inference(str(s1_best), 'stage1_test', imgsz=512, conf_thr=0.40)
run_inference(str(s2_best), 'stage2_test', imgsz=512, conf_thr=0.40)
run_inference(str(s3_best), 'stage3_test', imgsz=1280, conf_thr=0.25)

## Метрики качества

In [ ]:
def decode_gt_mask(lbl_path: Path, h: int, w: int) -> np.ndarray:
    mask = np.zeros((h, w), dtype=np.uint8)
    if not lbl_path.exists():
        return mask
    for ln in lbl_path.read_text(encoding='utf-8').splitlines():
        if not ln.strip():
            continue
        vals = list(map(float, ln.split()[1:]))
        if len(vals) < 6:
            continue
        pts = np.array(vals, dtype=np.float32).reshape(-1, 2)
        pts[:, 0] = (pts[:, 0] * w).clip(0, w - 1)
        pts[:, 1] = (pts[:, 1] * h).clip(0, h - 1)
        cv2.fillPoly(mask, [pts.astype(np.int32)], 1)
    return mask


def pred_to_mask(result, h: int, w: int) -> np.ndarray:
    mask = np.zeros((h, w), dtype=np.uint8)
    if result.masks is None:
        return mask
    m = result.masks.data
    union = m.any(dim=0).cpu().numpy().astype(np.uint8)
    return union


def compute_pixel_stats(gt: np.ndarray, pred: np.ndarray):
    gt_b = gt.astype(bool)
    pred_b = pred.astype(bool)
    tp = (gt_b & pred_b).sum()
    fp = (~gt_b & pred_b).sum()
    fn = (gt_b & ~pred_b).sum()

    union = tp + fp + fn
    iou = tp / union if union else 1.0
    prec = tp / (tp + fp) if tp + fp else 1.0
    rec = tp / (tp + fn) if tp + fn else 1.0

    diff = pred.astype(np.float32) - gt.astype(np.float32)
    rmse = float(np.sqrt(np.mean(diff ** 2)))

    return int(tp), int(fp), int(fn), float(iou), float(prec), float(rec), rmse

In [ ]:
def evaluate_model(weights, imgsz=512, conf_thr=0.25, nms_iou=0.3):
    mdl = YOLO(weights)

    val_img_pattern = str(YOLO_DS / 'images' / 'val' / '*.jpg')
    val_lbl_dir = YOLO_DS / 'labels' / 'val'

    records = []
    sum_tp = sum_fp = sum_fn = 0

    for r in mdl.predict(
        source=val_img_pattern,
        device=DEVICE,
        imgsz=imgsz,
        conf=conf_thr,
        iou=nms_iou,
        retina_masks=True,
        stream=True,
        save=False,
    ):
        img = cv2.imread(str(r.path))
        h, w = img.shape[:2]
        stem = Path(r.path).stem

        gt = decode_gt_mask(val_lbl_dir / f'{stem}.txt', h, w)
        pred = pred_to_mask(r, h, w)

        tp, fp, fn, iou, prec, rec, rmse = compute_pixel_stats(gt, pred)
        sum_tp += tp
        sum_fp += fp
        sum_fn += fn
        records.append({'file': Path(r.path).name, 'gt_px': int(gt.sum()),
                        'pred_px': int(pred.sum()), 'iou': iou,
                        'precision': prec, 'recall': rec, 'rmse': rmse})

    df = pd.DataFrame(records)
    global_iou = sum_tp / (sum_tp + sum_fp + sum_fn) if sum_tp + sum_fp + sum_fn else 1.0
    nonempty = df[df['gt_px'] > 0]

    summary = {
        'n_images': len(df),
        'global_iou': round(global_iou, 4),
        'mean_iou': round(df['iou'].mean(), 4),
        'mean_rmse': round(df['rmse'].mean(), 4),
        'iou_ge_0.5': round((nonempty['iou'] >= 0.5).mean(), 4),
        'iou_ge_0.75': round((nonempty['iou'] >= 0.75).mean(), 4),
        'iou_ge_0.9': round((nonempty['iou'] >= 0.9).mean(), 4),
    }
    return summary, df

In [ ]:
s1_metrics, _ = evaluate_model(str(s1_best), imgsz=512, conf_thr=0.40, nms_iou=0.3)
print('Stage 1:', s1_metrics)

In [ ]:
s2_metrics, _ = evaluate_model(str(s2_best), imgsz=512, conf_thr=0.40, nms_iou=0.3)
print('Stage 2:', s2_metrics)

In [ ]:
s3_metrics, _ = evaluate_model(str(s3_best), imgsz=1280, conf_thr=0.25, nms_iou=0.3)
print('Stage 3:', s3_metrics)

## Трекинг + ID Switches

In [ ]:

def box_iou(a, b):
    """IoU между двумя боксами [x1,y1,x2,y2] (нормализованными)."""
    ix1 = max(a[0], b[0]); iy1 = max(a[1], b[1])
    ix2 = min(a[2], b[2]); iy2 = min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = (a[2]-a[0]) * (a[3]-a[1])
    area_b = (b[2]-b[0]) * (b[3]-b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def run_tracker_with_idsw(weights, tracker_cfg, run_tag, imgsz=512, conf_thr=0.45, nms_iou=0.3):
    """
    Запускает трекинг и считает ID Switches.

    ID Switch: объект из предыдущего кадра (определяется по IoU >= 0.5)
    получает новый track_id в текущем кадре.
    """
    track_mdl = YOLO(weights)
    total_idsw = 0

    for vid in sorted(p for p in (Path('/kaggle/input/hw3-media') if IS_KAGGLE else Path('test_vids')).glob('*') if p.suffix.lower() in ('.mp4', '.mov', '.avi', '.mkv')):
        # prev: dict {track_id -> np.array([x1,y1,x2,y2]) normalized}
        prev_tracks = {}
        vid_idsw = 0

        gen = track_mdl.track(
            source=str(vid),
            device=DEVICE,
            imgsz=imgsz,
            conf=conf_thr,
            iou=nms_iou,
            tracker=tracker_cfg,
            persist=True,
            stream=True,
            save=True,
            project=str(Path('/kaggle/working/tracking_results') if IS_KAGGLE else Path('tracking_results')),
            name=run_tag,
            exist_ok=True,
            show=False,
            line_width=2,
            retina_masks=False,
        )

        for result in gen:
            if result.boxes is None or result.boxes.id is None:
                prev_tracks = {}
                continue

            curr_ids = result.boxes.id.cpu().numpy().astype(int)
            curr_xyxyn = result.boxes.xyxyn.cpu().numpy()  # нормализованные [x1,y1,x2,y2]
            curr_tracks = {int(tid): box for tid, box in zip(curr_ids, curr_xyxyn)}

            new_ids = set(curr_tracks.keys()) - set(prev_tracks.keys())
            for new_id in new_ids:
                new_box = curr_tracks[new_id]
                # Ищем совпадение с предыдущим треком по IoU
                for prev_id, prev_box in prev_tracks.items():
                    if prev_id not in curr_tracks and box_iou(new_box, prev_box) >= 0.5:
                        vid_idsw += 1
                        break

            prev_tracks = curr_tracks

        print(f'  {vid.name}: ID Switches = {vid_idsw}')
        total_idsw += vid_idsw

    print(f'Трекер {tracker_cfg!r} ({run_tag}): суммарно ID Switches = {total_idsw}')
    return total_idsw


In [ ]:

print('=== ByteTrack ===')
idsw_byte = run_tracker_with_idsw(str(s2_best), 'bytetrack.yaml', 'bytetrack_s2',
                                   imgsz=512, conf_thr=0.50)


In [ ]:

print('=== BoTSort ===')
idsw_bot = run_tracker_with_idsw(str(s2_best), 'botsort.yaml', 'botsort_s2',
                                  imgsz=512, conf_thr=0.50)

print()
print('Сводка ID Switches:')
print(f'  ByteTrack: {idsw_byte}')
print(f'  BoTSort:   {idsw_bot}')
print(f'  Лучший трекер: {"ByteTrack" if idsw_byte <= idsw_bot else "BoTSort"} (меньше = лучше)')
